# Rice Leaf Disease Classification: EfficientNet-B0 vs MobileNetV2

This notebook compares two transfer-learning models on the same rice leaf
disease dataset, using the same data split, the same image size and the
same training settings, so the comparison is fair.

**Dataset:** 5,932 images, 4 classes
- Bacterial Blight: 1,584
- Brown Spot: 1,600
- Blast: 1,440
- Tungro: 1,308

The dataset comes from Kaggle / Mendeley. Here it is loaded from its
Hugging Face mirror (`Project-AgML/rice_leaf_disease_classification_india`),
so no Kaggle account and no manual download is needed.

**Models:** EfficientNet-B0 and MobileNetV2, both pre-trained on ImageNet.

**How to run:** in Colab, set Runtime > Change runtime type > T4 GPU,
then run all cells from top to bottom. All figures and tables for the
paper are saved in Drive: `RICELEAF / result`.


## 1. Setup

Check the GPU, fix the random seed so the split is the same every time,
and set the folder where all results will be saved.


In [ ]:
import json
import random
import time
import copy
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# Fix the seed, so train / validation / test split is reproducible
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Connect Google Drive

All result images and tables are saved in `RICELEAF / result` in Drive,
so they can be used directly while writing the paper.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RESULT_DIR = Path("/content/drive/MyDrive/RICELEAF/result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("Results will be saved in:", RESULT_DIR)


## 3. Load the dataset

The dataset is loaded from Hugging Face. No file is uploaded by hand.


In [ ]:
!pip install -q datasets seaborn opencv-python-headless
from datasets import load_dataset

dataset = load_dataset("Project-AgML/rice_leaf_disease_classification_india")
print(dataset)
print("Classes:", dataset["train"].features["label"].names)
print("Images per class:", Counter(dataset["train"]["label"]))


## 4. Split the data

The data is split in a stratified way, so every class keeps the same
share in train, validation and test:

- Train: 70%
- Validation: 15%
- Test: 15%


In [ ]:
from datasets import DatasetDict

# First split off 30% for validation + test
train_and_rest = dataset["train"].train_test_split(
    test_size=0.30, seed=42, stratify_by_column="label"
)

# Then split that 30% equally into validation and test
val_and_test = train_and_rest["test"].train_test_split(
    test_size=0.50, seed=42, stratify_by_column="label"
)

splits = DatasetDict({
    "train": train_and_rest["train"],
    "val": val_and_test["train"],
    "test": val_and_test["test"],
})

for split_name in ["train", "val", "test"]:
    print(split_name, len(splits[split_name]),
          Counter(splits[split_name]["label"]))


## 5. Image preparation

Training images are augmented (crop, flip, rotation, colour change) so
the model does not memorise the training set. Validation and test images
are only resized and centre-cropped, with no augmentation.


In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class RiceLeafDataset(Dataset):
    # A small wrapper around the Hugging Face dataset.

    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]
        image = row["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(np.array(image))
        image = self.transform(image.convert("RGB"))
        return image, int(row["label"])


loaders = {
    "train": DataLoader(RiceLeafDataset(splits["train"], train_transform),
                        batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True),
    "val": DataLoader(RiceLeafDataset(splits["val"], eval_transform),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True),
    "test": DataLoader(RiceLeafDataset(splits["test"], eval_transform),
                       batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=2, pin_memory=True),
}

CLASS_NAMES = splits["train"].features["label"].names
print("Classes:", CLASS_NAMES)
print("Batches per split:", {k: len(v) for k, v in loaders.items()})


## 6. Build the two models

Both models start from ImageNet weights. Only the last layer is replaced,
because the original last layer was made for 1,000 ImageNet classes and
we have only 4 rice leaf classes.


In [ ]:
from torchvision import models


def build_model(model_name, num_classes=4):
    if model_name == "efficientnet_b0":
        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    return model


def count_parameters(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


for name in ["efficientnet_b0", "mobilenet_v2"]:
    model = build_model(name)
    print(name, f"{count_parameters(model):.2f} million parameters")


## 7. Train one model

The same function trains both models:

- Optimiser: AdamW, learning rate 0.0001
- Loss: CrossEntropyLoss with label smoothing 0.1
- Scheduler: cosine annealing over the full training
- Epochs: 12

After every epoch the model is checked on the validation set, and the
best validation model is kept. At the end it is tested once on the test
set, which the model has never seen during training.


In [ ]:
from sklearn.metrics import (accuracy_score,
                                 precision_recall_fscore_support,
                                 confusion_matrix,
                                 classification_report)

EPOCHS = 12
results = {}


def train_model(model_name, epochs=EPOCHS):
    print(f"\n===== Training {model_name} =====")
    model = build_model(model_name).to(device)

    loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimiser = torch.optim.AdamW(model.parameters(),
                                  lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=epochs)

    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_weights = None

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in loaders["train"]:
            images, labels = images.to(device), labels.to(device)
            optimiser.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimiser.step()

            total_loss += loss.item() * len(labels)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += len(labels)

        scheduler.step()

        # Validation accuracy for this epoch
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in loaders["val"]:
                outputs = model(images.to(device))
                val_correct += (outputs.argmax(1) == labels.to(device)).sum().item()
                val_total += len(labels)

        val_acc = val_correct / val_total
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch + 1}/{epochs} | "
              f"loss {total_loss / total:.4f} | "
              f"train acc {correct / total:.4f} | "
              f"val acc {val_acc:.4f} | "
              f"{time.time() - start_time:.0f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())

    # Load the best model and test it once
    model.load_state_dict(best_weights)
    torch.save(best_weights, RESULT_DIR / f"best_{model_name}.pth")

    model.eval()
    true_labels = []
    predicted_labels = []
    predicted_probs = []
    with torch.no_grad():
        for images, labels in loaders["test"]:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            predicted_probs.extend(probs.cpu().tolist())
            predicted_labels.extend(outputs.argmax(1).cpu().tolist())
            true_labels.extend(labels.tolist())

    test_acc = accuracy_score(true_labels, predicted_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="macro", zero_division=0)
    report = classification_report(true_labels, predicted_labels,
                                   target_names=CLASS_NAMES, digits=4)
    cm = confusion_matrix(true_labels, predicted_labels)

    print(f"\n{model_name}: best val acc {best_val_acc:.4f}, "
          f"test acc {test_acc:.4f}, precision {precision:.4f}, "
          f"recall {recall:.4f}, macro F1 {f1:.4f}")
    print(report)
    print("Confusion matrix (rows = true, columns = predicted):")
    print(cm)

    results[model_name] = {
        "best_val_acc": float(best_val_acc),
        "test_acc": float(test_acc),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "history": history,
        "confusion_matrix": cm.tolist(),
        "classification_report": report,
        "true_labels": true_labels,
        "predicted_labels": predicted_labels,
        "predicted_probs": predicted_probs,
        "classes": CLASS_NAMES,
    }
    return model


## 8. Train both models


In [ ]:
efficientnet_model = train_model("efficientnet_b0", epochs=EPOCHS)

In [ ]:
mobilenet_model = train_model("mobilenet_v2", epochs=EPOCHS)

## 9. Save the numbers

The test results are saved as a table and as JSON, so they can be
quoted directly in the paper.


In [ ]:
import csv

summary_rows = []
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "best_val_acc": round(r["best_val_acc"], 4),
        "test_acc": round(r["test_acc"], 4),
        "precision_macro": round(r["precision_macro"], 4),
        "recall_macro": round(r["recall_macro"], 4),
        "f1_macro": round(r["f1_macro"], 4),
    })
    with open(RESULT_DIR / f"classification_report_{name}.txt", "w") as f:
        f.write(r["classification_report"])

with open(RESULT_DIR / "results_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

with open(RESULT_DIR / "results.json", "w") as f:
    json.dump({k: {kk: vv for kk, vv in v.items()
                   if kk not in ("true_labels", "predicted_labels",
                                 "predicted_probs")}
               for k, v in results.items()}, f, indent=2)

for row in summary_rows:
    print(row)
print("\nSaved in:", RESULT_DIR)


## 10. Training curves

One accuracy / loss curve per model, plus one comparison figure that
puts both models on the same plot. These are the figures for the paper.


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 150

for name, r in results.items():
    history = r["history"]
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, history["train_acc"], marker="o", label="Train accuracy")
    ax1.plot(epochs, history["val_acc"], marker="s", label="Validation accuracy")
    ax1.set_title(f"{name} - Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, history["train_loss"], marker="o", color="tab:red")
    ax2.set_title(f"{name} - Training loss")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.grid(True)

    fig.tight_layout()
    fig.savefig(RESULT_DIR / f"{name}_training_curve.png", dpi=300)
    plt.show()

# Comparison: validation accuracy of both models on one plot
plt.figure(figsize=(8, 5))
for name, r in results.items():
    plt.plot(range(1, len(r["history"]["val_acc"]) + 1),
             r["history"]["val_acc"], marker="o", label=name)
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / "comparison_validation_accuracy.png", dpi=300)
plt.show()


## 11. Confusion matrices

One confusion matrix per model, with the counts written in each cell.


In [ ]:
import seaborn as sns

for name, r in results.items():
    cm = np.array(r["confusion_matrix"])
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {name} (test set)")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{name}_confusion_matrix.png", dpi=300)
    plt.show()


## 12. ROC curves

One ROC curve per class, per model, with the AUC value in the legend.


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_true_bin = label_binarize(
    results["efficientnet_b0"]["true_labels"], classes=[0, 1, 2, 3])

for name, r in results.items():
    y_score = np.array(r["predicted_probs"])
    plt.figure(figsize=(8, 6))
    for i, class_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        plt.plot(fpr, tpr, label=f"{class_name} (AUC = {auc(fpr, tpr):.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random guess")
    plt.title(f"ROC Curve - {name} (test set)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / f"{name}_roc_curve.png", dpi=300)
    plt.show()


## 13. Model comparison chart

Test accuracy and macro F1 of both models, side by side.


In [ ]:
model_names = list(results.keys())
test_accs = [results[m]["test_acc"] for m in model_names]
f1_scores = [results[m]["f1_macro"] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, test_accs, width, label="Test accuracy")
ax.bar(x + width / 2, f1_scores, width, label="Macro F1")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel("Score")
ax.set_title("EfficientNet-B0 vs MobileNetV2 (test set)")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", alpha=0.3)
for i, v in enumerate(test_accs):
    ax.text(i - width / 2, v + 0.02, f"{v:.4f}", ha="center", fontsize=9)
for i, v in enumerate(f1_scores):
    ax.text(i + width / 2, v + 0.02, f"{v:.4f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(RESULT_DIR / "comparison_accuracy_f1.png", dpi=300)
plt.show()


## 14. Sample predictions

Eight test images per model, with the true label and the predicted
label written above each image. This shows at a glance where the
model is right and where it is wrong.


In [ ]:
def show_sample_predictions(model, model_name, num_images=8):
    model.eval()
    test_dataset = RiceLeafDataset(splits["test"], eval_transform)
    indices = random.sample(range(len(test_dataset)), num_images)

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    for ax, idx in zip(axes, indices):
        image, true_label = test_dataset[idx]
        with torch.no_grad():
            pred_label = model(image.unsqueeze(0).to(device)).argmax(1).item()

        # Undo normalisation so the leaf looks normal again
        shown = image.cpu().numpy().transpose(1, 2, 0)
        shown = shown * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        shown = np.clip(shown, 0, 1)

        ax.imshow(shown)
        ax.axis("off")
        colour = "green" if pred_label == true_label else "red"
        ax.set_title(f"True: {CLASS_NAMES[true_label]}\nPred: {CLASS_NAMES[pred_label]}",
                     color=colour, fontsize=10)

    fig.suptitle(f"Sample Predictions - {model_name}", fontsize=14)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / f"{model_name}_sample_predictions.png", dpi=300)
    plt.show()


show_sample_predictions(efficientnet_model, "efficientnet_b0")
show_sample_predictions(mobilenet_model, "mobilenet_v2")


## 15. Grad-CAM

Grad-CAM shows which part of the leaf the model looked at when it made
its decision. One correctly predicted example is shown for each class,
for each model. Red means the model paid a lot of attention there.


In [ ]:
import cv2


def get_last_conv_layer(model, model_name):
    # The last convolutional block of the feature extractor
    return model.features[-1]


def make_gradcam(model, model_name, image_tensor, target_class):
    model.eval()
    activations = {}
    gradients = {}

    target_layer = get_last_conv_layer(model, model_name)

    def save_activation(module, inp, out):
        activations["value"] = out.detach()

    def save_gradient(module, grad_in, grad_out):
        gradients["value"] = grad_out[0].detach()

    handle_f = target_layer.register_forward_hook(save_activation)
    handle_b = target_layer.register_full_backward_hook(save_gradient)

    output = model(image_tensor.unsqueeze(0).to(device))
    model.zero_grad()
    output[0, target_class].backward()

    handle_f.remove()
    handle_b.remove()

    weights = gradients["value"].mean(dim=(2, 3), keepdim=True)
    heatmap = (weights * activations["value"]).sum(dim=1).squeeze()
    heatmap = torch.relu(heatmap).cpu().numpy()
    heatmap = heatmap / (heatmap.max() + 1e-8)
    heatmap = cv2.resize(heatmap, (IMAGE_SIZE, IMAGE_SIZE))
    return heatmap


def show_gradcam(model, model_name):
    model.eval()
    test_dataset = RiceLeafDataset(splits["test"], eval_transform)

    # Find one correct example for each class
    chosen = {}
    for idx in range(len(test_dataset)):
        image, true_label = test_dataset[idx]
        with torch.no_grad():
            pred_label = model(image.unsqueeze(0).to(device)).argmax(1).item()
        if true_label not in chosen and pred_label == true_label:
            chosen[true_label] = (image, idx)
        if len(chosen) == 4:
            break

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for col, class_id in enumerate(sorted(chosen.keys())):
        image, _ = chosen[class_id]
        heatmap = make_gradcam(model, model_name, image, class_id)

        original = image.cpu().numpy().transpose(1, 2, 0)
        original = original * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        original = np.clip(original, 0, 1)

        axes[0, col].imshow(original)
        axes[0, col].set_title(CLASS_NAMES[class_id], fontsize=11)
        axes[0, col].axis("off")

        coloured = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
        coloured = cv2.cvtColor(coloured, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.5 * original + 0.5 * coloured
        axes[1, col].imshow(np.clip(overlay, 0, 1))
        axes[1, col].set_title("Grad-CAM", fontsize=11)
        axes[1, col].axis("off")

    fig.suptitle(f"Grad-CAM - {model_name}", fontsize=14)
    fig.tight_layout()
    fig.savefig(RESULT_DIR / f"{model_name}_gradcam.png", dpi=300)
    plt.show()


show_gradcam(efficientnet_model, "efficientnet_b0")
show_gradcam(mobilenet_model, "mobilenet_v2")


## 16. Done

Everything for the paper is now in Drive: `RICELEAF / result`

- `results_summary.csv` and `results.json` — the numbers
- `classification_report_<model>.txt` — precision, recall, F1 per class
- `<model>_training_curve.png` — accuracy and loss over epochs
- `comparison_validation_accuracy.png` and `comparison_accuracy_f1.png`
- `<model>_confusion_matrix.png` and `<model>_roc_curve.png`
- `<model>_sample_predictions.png` and `<model>_gradcam.png`
- `best_<model>.pth` — the trained weights


In [ ]:
print("Files saved in:", RESULT_DIR)
for path in sorted(RESULT_DIR.glob("*")):
    print("-", path.name)
